# RAG и агенты для информационного поиска

Разрабатываем полноценную RAG-систему — от наивной версии до агента с tool use, который сам решает, когда использовать поиск, а когда отвечать напрямую.


## Установка зависимостей

Перед запуском нужно убедиться, что установлен Ollama и загружена модель:

> ```bash
> curl -fsSL https://ollama.com/install.sh | sh
> ollama pull qwen2.5:7b
> ```


#### Базовые библиотеки для RAG

In [ ]:
!pip install -q langchain langchain-community langchain-ollama langchain-huggingface langchain-text-splitters langchain-core langchain-qdrant
!pip install -q qdrant-client sentence-transformers rank_bm25
!pip install -q pypdf wikipedia tiktoken
!pip install -q matplotlib seaborn pandas numpy
!pip install -q ollama

In [ ]:
import os
import json
import time
import warnings
from typing import List, Dict, Any
from collections import defaultdict

import numpy as np
import pandas as pd

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama
from langchain_qdrant import QdrantVectorStore
from langchain_core.documents import Document

from sentence_transformers import CrossEncoder
from rank_bm25 import BM25Okapi


## 1. Подготовка корпуса документов

В реальном проекте корпус — это ваши данные: документация, регламенты, FAQ, база знаний. Для семинара возьмём документацию по популярным разделам Python, scikit-learn и PyTorch — близкая нам тематика, на которой легко проверять качество ответов.

### Технологии парсинга в индустрии

| Источник | Инструменты |
|---|---|
| PDF | PyPDF2, pdfplumber, PyMuPDF, **Unstructured.io**, **Docling** (IBM), LlamaParse |
| HTML | BeautifulSoup, **trafilatura**, **Firecrawl**, Crawl4AI |
| Office (DOCX/PPTX/XLSX) | python-docx, python-pptx, openpyxl, **Unstructured.io** |
| Сканы (OCR) | Tesseract, **EasyOCR**, **Yandex Vision OCR**, ABBYY FineReader |
| Vision LLMs | GPT-4o, Claude, Qwen-VL |


Для семинара используем заранее подготовленный мини-корпус.

В реальной системе на этом месте был бы код парсинга PDF/HTML.

In [ ]:
CORPUS = [
    {
        "title": "Python: List Comprehensions",
        "content": """List comprehension — это синтаксис для создания списков на основе итерируемых объектов.
Базовый синтаксис: [expression for item in iterable if condition].

Пример: [x**2 for x in range(10) if x % 2 == 0] создаёт список квадратов чётных чисел.

Преимущества list comprehension:
1. Более читаемый код по сравнению с циклом for и append
2. Часто работает быстрее, чем эквивалентный цикл
3. Возвращает полный список в памяти

Когда использовать: для коротких трансформаций, когда нужен весь список целиком.
Когда не использовать: для очень больших объёмов данных (используйте generator expression)."""
    },
    {
        "title": "Python: Generator Expressions",
        "content": """Generator expression — это ленивая (lazy) версия list comprehension.
Синтаксис идентичен, но используются круглые скобки: (x**2 for x in range(10)).

Главное отличие от list comprehension: generator не создаёт весь список в памяти,
а генерирует элементы по одному при итерации. Это критично для больших объёмов данных.

Пример экономии памяти:
- sum([x**2 for x in range(10**8)])  # выделяет память под 100 млн чисел
- sum(x**2 for x in range(10**8))    # вычисляет на лету, O(1) памяти

Generator может быть пройден только один раз. После полной итерации он "истощается".

Применение: чтение больших файлов, обработка потоковых данных, конвейеры (pipelines)."""
    },
    {
        "title": "Python: GIL (Global Interpreter Lock)",
        "content": """GIL (Global Interpreter Lock) — это мьютекс в CPython, защищающий доступ к Python-объектам.
GIL гарантирует, что только один поток выполняет байт-код Python в любой момент времени.

Последствия GIL:
1. Multi-threading в Python НЕ даёт параллельности для CPU-bound задач
2. Для I/O-bound задач (сеть, диск) threading работает эффективно — GIL отпускается во время блокирующих операций
3. Для CPU-bound задач используют multiprocessing — каждый процесс имеет свой GIL

Альтернативы для параллельных вычислений:
- multiprocessing: процессы вместо потоков
- concurrent.futures: высокоуровневый API
- numpy/pandas: операции в C-коде освобождают GIL
- asyncio: для I/O concurrency без потоков

В Python 3.13+ появилась экспериментальная сборка без GIL (free-threaded Python)."""
    },
    {
        "title": "Python: Threading vs Multiprocessing",
        "content": """threading и multiprocessing — два подхода к параллельному выполнению в Python.

threading (потоки):
- Общая память между потоками
- Лёгкие, быстрое создание
- Ограничены GIL для CPU-bound задач
- Хороши для I/O: сетевые запросы, чтение файлов

multiprocessing (процессы):
- Изолированная память, общение через IPC
- Тяжёлые, накладные расходы на создание
- Обходят GIL — реальная параллельность для CPU-bound
- Хороши для вычислений: numpy, обработка данных

Эмпирическое правило:
- Делаю много HTTP-запросов? → threading или asyncio
- Считаю числа, обрабатываю изображения? → multiprocessing
- Нужен максимум удобства? → concurrent.futures.ThreadPoolExecutor / ProcessPoolExecutor"""
    },
    {
        "title": "Python: Decorators",
        "content": """Декоратор — это функция, которая принимает другую функцию и расширяет её поведение
без изменения её кода. Используется синтаксис @decorator перед определением функции.

Базовый пример:
def my_decorator(func):
    def wrapper(*args, **kwargs):
        print("Before call")
        result = func(*args, **kwargs)
        print("After call")
        return result
    return wrapper

@my_decorator
def hello():
    print("Hello!")

Применения декораторов:
1. Логирование вызовов функций
2. Измерение времени выполнения (@timeit)
3. Кэширование результатов (@functools.lru_cache)
4. Аутентификация и авторизация в веб-фреймворках (Flask, Django)
5. Регистрация функций (например, в pytest, FastAPI)

Декораторы могут принимать аргументы — тогда нужен ещё один уровень вложенности функций."""
    },
    {
        "title": "Python: List vs Tuple",
        "content": """list и tuple — две основные коллекции в Python с похожим интерфейсом, но разными свойствами.

list (изменяемый, mutable):
- Создаётся через [] или list()
- Можно добавлять, удалять, изменять элементы
- Используется для коллекций однотипных элементов
- Чуть больший overhead памяти из-за резервирования места

tuple (неизменяемый, immutable):
- Создаётся через () или tuple()
- После создания изменить нельзя
- Используется для гетерогенных данных (например, кортеж координат (x, y))
- Может быть ключом словаря (list — не может)
- Чуть быстрее по доступу и потреблению памяти

Производительность:
- Создание tuple примерно в 2 раза быстрее list
- Доступ к элементам сравним
- Память: tuple занимает меньше места

Когда использовать tuple:
- Возврат нескольких значений из функции
- Координаты, RGB-цвета, фиксированные структуры
- Когда хочется защититься от случайного изменения данных"""
    },
    {
        "title": "scikit-learn: TF-IDF Vectorizer",
        "content": """TfidfVectorizer из sklearn.feature_extraction.text — классический инструмент
для преобразования текста в числовые признаки на основе частот слов.

TF-IDF (Term Frequency - Inverse Document Frequency) — мера важности слова в документе:
- TF: насколько часто слово встречается в данном документе
- IDF: насколько редко слово встречается в коллекции документов
- TF-IDF = TF * IDF: высокое значение для частых в документе, но редких в коллекции слов

Основные параметры TfidfVectorizer:
- max_features: ограничить размер словаря (например, 10000 самых частых)
- ngram_range: учитывать n-граммы, например (1,2) для unigram + bigram
- min_df, max_df: фильтрация слишком редких или слишком частых слов
- stop_words: список стоп-слов для удаления

Применение TF-IDF:
1. Классификация текстов (наивный Байес, SVM)
2. Кластеризация документов
3. Information retrieval (но BM25 обычно лучше)
4. Простой baseline перед нейросетевыми подходами"""
    },
    {
        "title": "scikit-learn: Random Forest",
        "content": """RandomForestClassifier и RandomForestRegressor — реализация ансамблевого метода
случайного леса в scikit-learn.

Случайный лес — это ансамбль решающих деревьев, обучаемых на бутстрэп-выборках с
случайным подмножеством признаков на каждом разбиении (bagging + feature randomness).

Ключевые параметры:
- n_estimators: количество деревьев (обычно 100-500)
- max_depth: глубина деревьев (None = полная глубина)
- min_samples_split, min_samples_leaf: контроль переобучения
- max_features: количество признаков для поиска лучшего разбиения
- n_jobs: параллелизация (-1 = все ядра)

Преимущества Random Forest:
1. Robust к шуму и выбросам
2. Естественно ранжирует важность признаков (feature_importances_)
3. Не требует масштабирования признаков
4. Работает с категориальными и числовыми признаками
5. Хорошо подходит как baseline для табличных данных

Недостатки:
- Хуже на разреженных данных высокой размерности
- Большая модель — медленный inference
- На некоторых задачах градиентный бустинг (XGBoost, LightGBM, CatBoost) точнее"""
    },
    {
        "title": "scikit-learn: Cross-Validation",
        "content": """Cross-validation (кросс-валидация) — техника оценки модели на разных разбиениях данных.

Базовая идея k-fold CV:
1. Разбить данные на k равных частей (folds)
2. Для каждой части: обучить модель на k-1 остальных, протестировать на этой
3. Усреднить метрики качества по всем k экспериментам

Зачем это нужно:
- Более надёжная оценка качества, чем одно train/test разбиение
- Снижает зависимость оценки от случайного разбиения
- Использует все данные и для обучения, и для валидации

Виды CV в scikit-learn:
- KFold: базовый k-fold
- StratifiedKFold: сохраняет баланс классов в каждом fold
- TimeSeriesSplit: для временных рядов (не нарушает порядок времени)
- GroupKFold: когда есть группы (например, пациенты), которые нельзя разделять
- LeaveOneOut: каждый объект — отдельный fold (дорого, для маленьких выборок)

Использование:
from sklearn.model_selection import cross_val_score
scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')"""
    },
    {
        "title": "scikit-learn: Pipelines",
        "content": """sklearn.pipeline.Pipeline — инструмент объединения нескольких шагов препроцессинга
и модели в единый объект, реализующий fit/predict.

Зачем нужны Pipelines:
1. Избегают data leakage: scaler/vectorizer обучается только на train fold
2. Атомарность: одно сохранение/загрузка вместо нескольких объектов
3. Чистый код: вместо последовательных вызовов — один pipeline.fit(X, y)
4. Совместимость с GridSearchCV: можно тюнить параметры всех шагов одновременно

Пример:
from sklearn.pipeline import Pipeline
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression())
])
pipe.fit(X_train, y_train)

Расширения:
- ColumnTransformer: разные преобразования для разных колонок
- FeatureUnion: параллельное применение трансформеров
- make_pipeline: упрощённый синтаксис без имён шагов

Best practice: всегда использовать Pipeline, даже если шагов всего два — это профилактика
data leakage и упрощает CV."""
    },
    {
        "title": "PyTorch: Autograd",
        "content": """Autograd — система автоматического дифференцирования в PyTorch.
Это ядро, обеспечивающее backpropagation в нейросетях.

Принцип работы:
1. PyTorch строит динамический вычислительный граф во время forward pass
2. Каждая операция запоминает свою функцию обратного градиента
3. Вызов loss.backward() запускает обратное распространение
4. Градиенты накапливаются в .grad атрибутах тензоров с requires_grad=True

Ключевые концепты:
- requires_grad: флаг тензора, отслеживать ли градиенты
- .backward(): запуск обратного распространения от данного тензора
- .grad: накопленные градиенты тензора
- torch.no_grad(): контекст для отключения отслеживания (inference, eval)
- detach(): создание тензора без связи с графом

Типичный training loop:
for x, y in dataloader:
    optimizer.zero_grad()  # обнуляем градиенты
    pred = model(x)
    loss = criterion(pred, y)
    loss.backward()        # вычисляем градиенты
    optimizer.step()       # обновляем параметры

Динамический vs статический граф:
- PyTorch: динамический (define-by-run) — гибче, проще для исследований
- TensorFlow 1.x был статическим (define-then-run); TF 2 перешёл на eager execution"""
    },
    {
        "title": "PyTorch: DataLoader",
        "content": """torch.utils.data.DataLoader — стандартный инструмент для работы с батчами данных в PyTorch.

DataLoader оборачивает Dataset и предоставляет:
- Батчинг: сбор отдельных примеров в батчи
- Shuffle: перемешивание данных каждую эпоху
- Параллельную загрузку: несколько worker-процессов
- Collation: кастомное объединение примеров в батч

Базовое использование:
from torch.utils.data import DataLoader
loader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=4)
for batch in loader:
    ...

Параметры:
- batch_size: размер батча (обычно 32-256, зависит от GPU)
- shuffle: True для train, False для val/test
- num_workers: количество процессов для параллельной загрузки (на CPU)
- pin_memory: True ускоряет передачу на GPU
- collate_fn: кастомная функция объединения (для текстов с padding)
- drop_last: отбросить неполный последний батч

Best practices:
- Для CPU-bound препроцессинга: num_workers > 0
- Для GPU-обучения: pin_memory=True, num_workers=4-8
- Для больших датасетов: использовать IterableDataset вместо MapDataset"""
    },
]

print(f"Заголовок: {CORPUS[0]['title']}")
print(f"Содержимое: {CORPUS[0]['content'][:200]}...")

## 2. Naive RAG: минимальная версия

Начнём с самой простой версии. Пайплайн:
1. Разбить документы на чанки (chunking)
2. Получить эмбеддинги каждого чанка
3. Сохранить в vector store
4. На запрос: эмбеддинг → top-k поиск → промпт с контекстом → LLM

### 2.1 Chunking — разбиение на фрагменты

**Стратегии в индустрии:**
- **Fixed-size** (`CharacterTextSplitter`) — бейзлайн
- **Recursive** (`RecursiveCharacterTextSplitter`) — стандарт индустрии, учитывает структуру (абзацы → предложения → слова)
- **Document-aware** (`MarkdownHeaderTextSplitter`, `HTMLHeaderTextSplitter`) — по структуре документа
- **Semantic** (`SemanticChunker`) — по эмбеддингам соседних предложений
- **Late chunking** (Jina, 2024) — эмбеддить весь документ, потом нарезать
- **Contextual Retrieval** — добавлять LLM-сгенерированный контекст к каждому чанку

Используем `RecursiveCharacterTextSplitter` — самый популярный выбор для baseline.

In [ ]:
documents = [
    Document(page_content=doc["content"], metadata={"title": doc["title"], "doc_id": i})
    for i, doc in enumerate(CORPUS)
]

splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,  # размер в символах (для русского это ~150 токенов)
    chunk_overlap=80,  # размер в символах - 20% overlap от chunk_size для сохранения контекста на границах
    separators=["\n\n", "\n", ". ", " ", ""]  # иерархия разделителей
)

chunks = splitter.split_documents(documents)


In [ ]:
print(f"Получили {len(chunks)} чанков из {len(documents)} документов")
print(f"Метаданные: {chunks[0].metadata}")
print(f"Текст: {chunks[0].page_content[:200]}...")

### 2.2 Embeddings — векторизация чанков

Используем `intfloat/multilingual-e5-large`

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-large",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)


In [ ]:
sample_text = "Что такое декоратор в Python?"
sample_vector = embeddings.embed_query(sample_text)

print(f"Текст: '{sample_text}'")
print(f"Размерность вектора: {len(sample_vector)}")
print(f"Первые 10 значений: {[round(x, 3) for x in sample_vector[:10]]}")
print(f"Норма вектора (после normalize): {np.linalg.norm(sample_vector):.3f}")

### 2.3 Vector Store — хранение и поиск

Используем Qdrant в in-memory режиме

#### Этап offline-indexing

In [ ]:
start = time.time()

vector_store = QdrantVectorStore.from_documents(
    chunks,
    embeddings,
    location=":memory:",
    collection_name="docs",
)

elapsed = time.time() - start
print(f"Индекс построен за {elapsed:.1f} сек ({len(chunks)} чанков)")

### 2.4 Retrieval + Generation — собираем pipeline

Что используем для генерации:
- Локально через Ollama: Qwen2.5 7B
- Альтернативы: Llama 3.2, Mistral, Phi-3, Gemma 2

Подключаем локальную LLM через Ollama

Перед запуском должна быть выполнена команда: `ollama pull qwen2.5:7b`

In [ ]:
llm = ChatOllama(
    model="qwen2.5:7b",
    temperature=0.0,
    num_predict=1024,
)


In [ ]:
test_response = llm.invoke("Скажи 'привет' одним словом").content
print(f"Ответ LLM: {test_response}")

In [ ]:
def naive_rag(query: str, k: int = 5):
    
    retrieved = vector_store.similarity_search(query, k=k)
    
    context = "\n\n".join(
        [
            f"[Источник {i+1}: {doc.metadata['title']}]\n{doc.page_content}"
            for i, doc in enumerate(retrieved)
        ]
    )
    
    prompt = f"""Ответь на вопрос пользователя на основе предоставленного контекста.
Если ответа в контексте нет — скажи, что не знаешь. Указывай номер источника в формате [N].

Контекст:
{context}

Вопрос: {query}

Ответ:"""
    
    answer = llm.invoke(prompt).content
    
    return answer, retrieved


In [ ]:
question = "Чем list comprehension отличается от generator expression?"
answer, retrieved = naive_rag(question)

print(f"Вопрос: {question}\n")
print(f"Найденные источники (top-5):")
for i, doc in enumerate(retrieved):
    print(f"  {i+1}. {doc.metadata['title']}")
print(f"Ответ:\n{answer}")

## 3. Анализ ошибок Naive RAG

Прогоним несколько проблемных запросов, на которых naive RAG начинает плыть.

In [ ]:
PROBLEMATIC_CASES = [
    {
        "query": "как сделать чтобы код в питоне работал параллельно",
        "issue": "Vocabulary mismatch: разговорный язык vs терминология документации",
        "should_find": ["GIL", "Threading vs Multiprocessing"]
    },
    {
        "query": "что такое GIL",
        "issue": "Точная аббревиатура — dense retrieval может промахнуться",
        "should_find": ["GIL"]
    },
    {
        "query": "сравни list и tuple по производительности",
        "issue": "Сравнительный запрос — нужен один документ с обеими сущностями",
        "should_find": ["List vs Tuple"]
    },
    {
        "query": "как обучить нейросеть для предсказания цен на квартиры",
        "issue": "Out-of-scope: ответа в корпусе нет",
        "should_find": []
    },
]

def show_retrieval(query, k=5):
    retrieved = vector_store.similarity_search(query, k=k)
    print(f"\n{'='*70}")
    print(f"Query: {query}")
    print(f"{'='*70}")
    print(f"Top-{k} найденных чанков:")
    for i, doc in enumerate(retrieved):
        title = doc.metadata['title']
        snippet = doc.page_content[:80].replace('\n', ' ')
        print(f"  {i+1}. [{title}] {snippet}...")
    return retrieved

for case in PROBLEMATIC_CASES:
    print(f"\n\n\nПроблема: {case['issue']}")
    if case['should_find']:
        print(f"   Ожидаем найти: {', '.join(case['should_find'])}")
    show_retrieval(case['query'])

**Что мы видим:**

1. **Vocabulary mismatch:** для разговорного "как сделать чтобы код работал параллельно" может находиться не самое релевантное — нужна модель, которая лучше связывает разговорный и формальный язык, или query rewriting
2. **Точная аббревиатура GIL:** dense retrieval хорошо работает с семантикой, но точные совпадения слов лучше ловит BM25 — отсюда необходимость hybrid search
3. **Сравнительный запрос:** нужны конкретные техники типа query decomposition или multi-vector retrieval
4. **Out-of-scope:** базовый pipeline всё равно "найдёт" что-то нерелевантное — нужны порог релевантности или CRAG с web search fallback

Теперь — каждое улучшение в действии.

## 4. Hybrid Search: BM25 + Dense

Объединяем плотный (semantic) и разреженный (lexical) поиск. Каждый ловит то, что другой пропускает.

### 4.1 BM25 — что это и почему важно

BM25 — классический алгоритм ранжирования из IR, основанный на статистике слов

Когда BM25 побеждает dense:
- Точные термины, аббревиатуры (GIL, TF-IDF, FZ-152)
- Имена собственные, числа, идентификаторы
- Out-of-domain слова, которых модель не видела при обучении

### 4.2 Reciprocal Rank Fusion (RRF) — как объединять

RRF — простой и эффективный способ объединить несколько ранжированных списков.

Формула: `score(d) = Σ 1 / (k + rank_i(d))` по всем источникам

где `k` — константа, `rank_i(d)` — позиция документа в i-м ранжировании.


In [ ]:
def simple_tokenize(text):
    return text.lower().split()

tokenized_chunks = [simple_tokenize(chunk.page_content) for chunk in chunks]

bm25 = BM25Okapi(tokenized_chunks)

print(f"Размер словаря: {len(bm25.idf)} уникальных токенов")

In [ ]:
def bm25_search(query: str, k: int = 10):
    tokens = simple_tokenize(query)
    scores = bm25.get_scores(tokens)
    top_indices = scores.argsort()[-k:][::-1]
    return [(chunks[i], scores[i]) for i in top_indices]

query = "что такое GIL"
print(f"Запрос: '{query}'\n")

print("Dense retrieval (top-3):")
dense_results = vector_store.similarity_search(query, k=3)
for i, doc in enumerate(dense_results):
    print(f"  {i+1}. [{doc.metadata['title']}]")

print("\nBM25 retrieval (top-3):")
bm25_results = bm25_search(query, k=3)
for i, (doc, score) in enumerate(bm25_results):
    print(f"  {i+1}. [{doc.metadata['title']}] (score={score:.2f})")

In [ ]:
def rrf_fusion(rankings: List[List[Document]], k: int = 60) -> List[tuple]:
    scores = defaultdict(float)
    docs_by_id = {}
    
    for ranking in rankings:
        for rank, doc in enumerate(ranking):
            doc_id = doc.page_content[:100]
            scores[doc_id] += 1.0 / (k + rank + 1)
            docs_by_id[doc_id] = doc
    
    sorted_ids = sorted(scores.keys(), key=lambda x: -scores[x])
    return [(docs_by_id[doc_id], scores[doc_id]) for doc_id in sorted_ids]

def hybrid_search(query: str, k: int = 10, candidates: int = 20):
    dense_results = vector_store.similarity_search(query, k=candidates)
    bm25_results = [doc for doc, _ in bm25_search(query, k=candidates)]
    
    fused = rrf_fusion([dense_results, bm25_results])
    return [doc for doc, _ in fused[:k]]


In [ ]:
query = "что такое GIL"
print(f"Запрос: '{query}'\n")

hybrid_results = hybrid_search(query, k=5)
print("Hybrid retrieval (top-5):")
for i, doc in enumerate(hybrid_results):
    print(f"  {i+1}. [{doc.metadata['title']}]")
    print(f"     {doc.page_content[:100].replace(chr(10), ' ')}...")

In [ ]:
comparison_queries = [
    "что такое GIL",
    "как работают декораторы",
    "TF-IDF",
    "сравни list и tuple",
]

print("Сравнение Dense vs Hybrid retrieval:")
print("="*70)

for q in comparison_queries:
    print(f"\nQuery: {q}")
    
    dense = vector_store.similarity_search(q, k=3)
    hybrid = hybrid_search(q, k=3)
    
    print("  Dense:")
    for doc in dense:
        print(f"    • {doc.metadata['title']}")
    print("  Hybrid:")
    for doc in hybrid:
        print(f"    • {doc.metadata['title']}")

**В production почти всегда выбирают hybrid**. Современные vector DB (Qdrant, Weaviate, Elasticsearch) поддерживают hybrid из коробки одним запросом.

## 5. Reranking: точное переранжирование top-N

После hybrid search мы получаем 20-30 кандидатов хорошего качества. Теперь хочется выбрать самые релевантные для финального промпта. Это работа cross-encoder reranker.

### 5.1 Bi-encoder vs Cross-encoder

Bi-encoder (то, что у нас сейчас в dense retrieval):
- Запрос → вектор. Документ → вектор. Сравнение через cosine.
- Быстро, можно индексировать заранее, плохо для тонких различий

Cross-encoder:
- Запрос + документ → BERT → score
- Точно, но медленно: O(N) для каждого запроса
- Идеально для переранжирования небольшого числа кандидатов

### 5.2 Архитектурный паттерн "retrieve-then-rerank"

```
Query → Hybrid Retriever (top-30, быстро) → Cross-encoder Rerank (top-5, точно) → LLM
```

Это стандарт в современном production-RAG.

Используем `BAAI/bge-reranker-v2-m3`

In [ ]:
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", max_length=512)

In [ ]:
def rerank(query: str, documents: List[Document], top_k: int = 5):
    if not documents:
        return []
    
    pairs = [[query, doc.page_content] for doc in documents]
    
    scores = reranker.predict(pairs, show_progress_bar=False)
    
    ranked = sorted(zip(documents, scores), key=lambda x: -x[1])
    return ranked[:top_k]

query = "сравни list и tuple по производительности"
print(f"Query: '{query}'\n")

candidates = hybrid_search(query, k=10)

print("До reranking (hybrid top-10):")
for i, doc in enumerate(candidates):
    print(f"  {i+1}. {doc.metadata['title']}")

print("\nПосле reranking (cross-encoder top-5):")
reranked = rerank(query, candidates, top_k=5)
for i, (doc, score) in enumerate(reranked):
    print(f"  {i+1}. {doc.metadata['title']} (rerank_score={score:.3f})")

In [ ]:
def advanced_rag(query: str, retrieve_k: int = 20, rerank_k: int = 5):
    
    candidates = hybrid_search(query, k=retrieve_k)
    
    reranked = rerank(query, candidates, top_k=rerank_k)
    top_chunks = [doc for doc, _ in reranked]
    
    context = "\n\n".join([
        f"[Источник {i+1}: {doc.metadata['title']}]\n{doc.page_content}"
        for i, doc in enumerate(top_chunks)
    ])
    
    prompt = f"""Ответь на вопрос пользователя на основе предоставленного контекста.
Если ответа в контексте нет — скажи, что не знаешь. Указывай номер источника в формате [N].

Контекст:
{context}

Вопрос: {query}

Ответ:"""
    
    answer = llm.invoke(prompt).content
    return answer, top_chunks


In [ ]:
question = "сравни list и tuple по производительности"
answer, retrieved = advanced_rag(question)

print(f"Query: {question}\n")
print(f"Top-5 после rerank:")
for i, doc in enumerate(retrieved):
    print(f"  {i+1}. {doc.metadata['title']}")
print(f"\nОтвет:\n{answer}")

## 6. Метрики качества: измеряем улучшение объективно

### 6.1 Что измеряем — ключевые RAG-метрики

| Метрика | Что измеряет |
|---|---|
| **Faithfulness (groundedness)** | Опирается ли ответ на контекст |
| **Answer Relevance** | Отвечает ли ответ на вопрос |
| **Context Precision** | Релевантны ли найденные чанки |
| **Context Recall** | Покрыт ли весь нужный материал |
| **Answer Correctness** | Правильный ли ответ |

### 6.2 LLM-as-a-Judge

Используем сам LLM как "судью" для оценки faithfulness. Это стандарт сегодня — корреляция с human judgment 0.7-0.85, что выше любых классических метрик типа BLEU/ROUGE.

In [ ]:
def evaluate_faithfulness(query: str, answer: str, retrieved_chunks: List[Document]) -> int:
    
    context = "\n\n".join([f"[{i+1}] {c.page_content}" for i, c in enumerate(retrieved_chunks)])
    
    judge_prompt = f"""Ты оцениваешь качество ответа RAG-системы.

Контекст, который был дан системе:
{context}

Вопрос пользователя: {query}

Ответ системы: {answer}

Оцени ВЕРНОСТЬ ответа контексту (faithfulness): все ли утверждения в ответе следуют 
из предоставленного контекста?

Шкала:
5 — все утверждения в ответе полностью подтверждаются контекстом
4 — почти все утверждения подтверждаются, есть незначительные дополнения
3 — основные утверждения подтверждаются, но есть и не подтверждённые
2 — значительная часть ответа не опирается на контекст
1 — ответ почти не связан с контекстом или галлюцинирует

Ответь ТОЛЬКО одной цифрой от 1 до 5, без пояснений."""
    
    response = llm.invoke(judge_prompt).content.strip()
    
    for char in response:
        if char.isdigit(): # Извлекаем первую цифру
            score = int(char)
            return score if 1 <= score <= 5 else 3
    return 3


def evaluate_answer_relevance(query: str, answer: str) -> int:
    judge_prompt = f"""Оцени, насколько ответ системы соответствует заданному вопросу.

Вопрос: {query}
Ответ: {answer}

Шкала:
5 — ответ полностью и прямо отвечает на вопрос
4 — ответ отвечает на вопрос, но с незначительными отклонениями
3 — ответ частично отвечает на вопрос
2 — ответ слабо связан с вопросом
1 — ответ не отвечает на вопрос

Ответь ТОЛЬКО одной цифрой от 1 до 5."""
    
    response = llm.invoke(judge_prompt).content.strip()
    for char in response:
        if char.isdigit():
            score = int(char)
            return score if 1 <= score <= 5 else 3
    return 3


In [ ]:
EVAL_QUESTIONS = [
    "Что такое декоратор в Python и зачем он нужен?",
    "Чем list comprehension отличается от generator expression?",
    "Что такое GIL и как он влияет на многопоточность?",
    "Когда использовать threading, а когда multiprocessing?",
    "Как работает RandomForest в sklearn?",
    "Зачем нужны Pipelines в scikit-learn?",
    "Что такое cross-validation?",
    "Как работает autograd в PyTorch?",
    "Что такое DataLoader и зачем нужны workers?",
    "Чем отличается list от tuple по памяти и производительности?",
]


In [ ]:
results = []
for i, question in enumerate(EVAL_QUESTIONS):
    print(f"  [{i+1}/{len(EVAL_QUESTIONS)}] {question[:60]}...")
    
    a1, r1 = naive_rag(question)
    naive_faith = evaluate_faithfulness(question, a1, r1)
    naive_rel = evaluate_answer_relevance(question, a1)
    
    a2, r2 = advanced_rag(question)
    adv_faith = evaluate_faithfulness(question, a2, r2)
    adv_rel = evaluate_answer_relevance(question, a2)
    
    results.append({
        "question": question,
        "naive_faithfulness": naive_faith,
        "naive_relevance": naive_rel,
        "advanced_faithfulness": adv_faith,
        "advanced_relevance": adv_rel,
    })

results_df = pd.DataFrame(results)
print(results_df.describe()[['naive_faithfulness', 'naive_relevance', 
                               'advanced_faithfulness', 'advanced_relevance']].round(2))

**Ключевая мысль:** теперь у нас есть способ сравнивать версии pipeline. В production:
- Eval запускается в CI на каждое изменение
- A/B-тесты на реальных пользователях для финального решения о внедрении


## 7. От RAG к агенту: tool use и ReAct

Текущая система — статический pipeline: всегда retrieve, всегда generate. Но не все запросы требуют поиска. Превратим RAG в инструмент агента, который сам решает, когда его использовать.

### 7.1 Концепция

**ReAct (Reasoning + Acting)** — фундаментальная парадигма агентов: чередование рассуждения и действий.

```
User: Что такое декоратор?
Agent: [Thought] Это вопрос о Python, нужен поиск в документации
       [Action] search_docs("декоратор Python")
       [Observation] Декоратор — это функция, которая...
       [Thought] У меня есть нужная информация
       [Answer] Декоратор в Python — это...

User: Сколько будет 2^32?
Agent: [Thought] Это вычисление, поиск не нужен
       [Action] calculate("2**32")
       [Observation] 4294967296
       [Answer] 2^32 = 4 294 967 296

User: Привет!
Agent: [Thought] Простое приветствие, инструменты не нужны
       [Answer] Привет! Чем могу помочь?
```

### 7.2 Технологии для построения агентов

| Фреймворк | Особенности |
|---|---|
| **LangChain Agents** | Самый популярный, легко начать |
| **LangGraph** | Граф состояний, более низкоуровневый и мощный |
| **LlamaIndex Agents** | Хорошо интегрирован с RAG-функциональностью |
| **CrewAI** | Multi-agent через "роли в команде" |
| **AutoGen** (Microsoft) | Multi-agent через диалог |
| **OpenAI Agents SDK** | Лёгкий, нативный для OpenAI tool use |
| **Haystack Agents** | От deepset, для enterprise |

Современные LLM (GPT-4o, Claude, Qwen2.5, Gemini, YandexGPT 5, GigaChat) поддерживают native function calling — структурированный вызов функций без хрупкого промптинга.

In [ ]:
def search_documentation(query: str) -> str:
    """Поиск в документации Python и ML библиотек.
    
    Используется для вопросов о Python, scikit-learn, PyTorch, программировании.
    Возвращает релевантные фрагменты документации.
    """
    
    candidates = hybrid_search(query, k=10)
    reranked = rerank(query, candidates, top_k=3)
    
    if not reranked:
        return "Ничего не найдено в документации."
    
    result = "Найденные фрагменты документации:\n\n"
    for i, (doc, score) in enumerate(reranked):
        result += f"[Источник {i+1}: {doc.metadata['title']}]\n{doc.page_content}\n\n"
    return result.strip()


def calculate(expression: str) -> str:
    """Вычисляет математическое выражение.
    
    Поддерживает: +, -, *, /, **, %, скобки.
    Пример: "2**32 + 100"
    """
    try:
        allowed = {"__builtins__": {}}
        result = eval(expression, allowed)
        return f"Результат: {result}"
    except Exception as e:
        return f"Ошибка вычисления: {e}"


TOOLS = {
    "search_documentation": {
        "function": search_documentation,
        "description": "Поиск в документации Python и ML библиотек. Используется для технических вопросов о Python, sklearn, PyTorch.",
        "parameters": {"query": "Поисковый запрос на русском или английском"}
    },
    "calculate": {
        "function": calculate,
        "description": "Вычислить математическое выражение. Поддерживает арифметику и возведение в степень.",
        "parameters": {"expression": "Математическое выражение, например '2**32'"}
    }
}

print("Tool-ы определены:")
for name, info in TOOLS.items():
    print(f"  {name}: {info['description'][:60]}...")

In [ ]:
import re

def format_tools_description():
    desc = "Доступные инструменты:\n\n"
    for name, info in TOOLS.items():
        params = ", ".join(f"{k}: {v}" for k, v in info["parameters"].items())
        desc += f" {name}({params})\n  {info['description']}\n\n"
    return desc


SYSTEM_PROMPT = f"""Ты — помощник, который отвечает на вопросы пользователей.
У тебя есть инструменты, которые можно и нужно использовать.

{format_tools_description()}

Чтобы вызвать инструмент, используй формат:
TOOL_CALL: имя_инструмента(параметр="значение")

Обязательно обращайся к инструментам, если они могут помочь при ответе на вопрос, в этом случае не генерируй ответ сам.
Если инструмент не нужен — просто отвечай напрямую.
После получения результата инструмента ты можешь либо вызвать ещё один, либо дать финальный ответ.

Финальный ответ обязательно начинай с фразы FINAL_ANSWER:
"""


def parse_tool_call(text: str):
    """Парсит строку TOOL_CALL: name(param="value") → (name, params dict)."""
    match = re.search(r'TOOL_CALL:\s*(\w+)\((.*?)\)', text, re.DOTALL)
    if not match:
        return None
    
    name = match.group(1)
    args_str = match.group(2)
    
    params = {}
    for arg_match in re.finditer(r'(\w+)\s*=\s*"([^"]*)"', args_str):
        params[arg_match.group(1)] = arg_match.group(2)
    
    return name, params


def run_agent(user_query: str, max_steps: int = 5, verbose: bool = True):
    """
    Запускает ReAct-цикл: модель думает → вызывает tool → получает результат → ...
    """
    history = [
        f"SYSTEM: {SYSTEM_PROMPT}",
        f"USER: {user_query}"
    ]
    
    for step in range(max_steps):
        if verbose:
            print(f"\n--- Шаг {step + 1} ---")
        
        prompt = "\n\n".join(history) + "\n\nASSISTANT:"
        
        response = llm.invoke(prompt).content
        
        if verbose:
            print(f" Модель:\n{response[:500]}")
        
        if "FINAL_ANSWER:" in response:
            final = response.split("FINAL_ANSWER:")[1].strip()
            if verbose:
                print(f"\nФинальный ответ получен")
            return final, history
        
        tool_call = parse_tool_call(response)  # Иначе — ищем tool call
        if tool_call is None:
            if verbose:
                print(f"\n!!! Модель не вызвала tool и не пометила FINAL_ANSWER, возвращаем как есть")
            return response.strip(), history
        
        name, params = tool_call
        if name not in TOOLS:
            history.append(f"ASSISTANT: {response}")
            history.append(f"OBSERVATION: Ошибка - инструмент '{name}' не существует")
            continue
        
        if verbose:
            print(f"\n Вызов: {name}({params})")
        
        try:
            result = TOOLS[name]["function"](**params)
        except Exception as e:
            result = f"Ошибка выполнения: {e}"
        
        if verbose:
            preview = result[:200] + "..." if len(result) > 200 else result
            print(f" Результат:\n{preview}")
        
        history.append(f"ASSISTANT: {response}")
        history.append(f"OBSERVATION: {result}")
    
    return "Превышено максимальное число шагов агента", history


##### Кейс A: вопрос требует поиска в документации

In [ ]:
print("="*70)
print("КЕЙС A: Технический вопрос → агент должен использовать search_documentation")
print("="*70)
answer_a, history_a = run_agent("Что такое GIL в Python?", verbose=True)
print(f"\n ИТОГ: {answer_a}")

In [ ]:
print("="*70)
print("КЕЙС A: Технический вопрос → агент должен использовать search_documentation")
print("="*70)
answer_a, history_a = run_agent("What is GIL in Python?", verbose=True)
print(f"\n ИТОГ: {answer_a}")

##### Кейс B: математическое вычисление

In [ ]:
print("="*70)
print("КЕЙС B: Вычисление → агент должен использовать calculate")
print("="*70)
answer_b, history_b = run_agent("Сколько будет 2 в степени 32?", verbose=True)
print(f"\n ИТОГ: {answer_b}")

##### Кейс C: общий вопрос без необходимости в инструментах

In [ ]:
print("="*70)
print("КЕЙС C: Простое приветствие → агент не должен вызывать tools")
print("="*70)
answer_c, history_c = run_agent("Привет, как дела?", verbose=True)
print(f"\n ИТОГ: {answer_c}")

##### Кейс D: составной вопрос — поиск + вычисление

In [ ]:
print("="*70)
print("КЕЙС D: Составной вопрос → агент использует оба инструмента")
print("="*70)
answer_d, history_d = run_agent(
    "Если у меня список из 1000 элементов в Python, сколько примерно памяти он займёт? "
    "Считай, что каждый элемент — это int, занимающий 28 байт.",
    verbose=True
)
print(f"\n ИТОГ: {answer_d}")

In [ ]:
print("="*70)
print("КЕЙС E: Составной вопрос → агент должен использовать search_documentation")
print("="*70)
answer_e, history_e = run_agent(
    "How to use DataLoader?",
    verbose=True
)
print(f"\n ИТОГ: {answer_d}")

**В production для агентов используют:**
- **LangGraph** — для сложных workflows с состояниями и условными переходами
- **Native function calling API** (OpenAI, Anthropic, Google) — структурированный, надёжный, parallel tool calls
- **Observability** (Langfuse, LangSmith) — критично для отладки длинных траекторий
- **Guardrails** (NeMo Guardrails, LlamaGuard) — защита от prompt injection через retrieved data

## 8. Overview

| # | Техника | Технологии в коде | Production-альтернативы |
|---|---|---|---|
| 1 | Document loading | hardcoded list | Unstructured, Docling, LlamaParse |
| 2 | Chunking | RecursiveCharacterTextSplitter | + Document-aware, Semantic, Late chunking, Contextual Retrieval |
| 3 | Embeddings | multilingual-e5-large | OpenAI, Cohere, Voyage |
| 4 | Vector store | Qdrant in-memory | Qdrant managed, Weaviate, Milvus, pgvector |
| 5 | Sparse retrieval | rank_bm25 | Elasticsearch, OpenSearch, Vespa |
| 6 | Hybrid + RRF | руками | Native в Qdrant/Weaviate/Elasticsearch |
| 7 | Reranking | bge-reranker-v2-m3 | Cohere Rerank, Jina, Voyage |
| 8 | Generation | Qwen2.5 via Ollama | OpenAI, Anthropic, YandexGPT, GigaChat |
| 9 | Evaluation | LLM-as-judge руками | RAGAS, DeepEval, Langfuse, TruLens |
| 10 | Agent | ReAct руками | LangGraph, LangChain Agents, OpenAI Agents SDK |

N.B.: Все, что мы построили, расширяемо другими техниками, рассмотренными на лекции